# scBriDi: Bridging Alignment for Cross-Modal Multi-Omics Generation

We propose a cross-modal generative framework built around a transferable, **RNA-centered Bridging Alignment** strategy. Leveraging the richer and more comprehensive information content of scRNA-seq, RNA serves as the "semantic center" that connects other omic modalities. The framework is trained in progressive stages.


In [1]:
import sys
sys.path.append('..')
from src.scBriDi import scBriDi
import numpy as np
import torch
import scanpy as sc
from src.util import five_fold_split_dataset,cluster_metrics

d:\App\Anaconda\Lib\site-packages\scanpy\_utils\__init__.py:35: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
d:\App\Anaconda\Lib\site-packages\scanpy\__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
d:\App\Anaconda\Lib\site-packages\scanpy\readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [ ]:
tdat = sc.read_h5ad('../data/BMMC/ADT_2.h5ad')
rdat1 = sc.read_h5ad('../data/BMMC/GEX_c1.h5ad')
rdat2 = sc.read_h5ad('../data/BMMC/GEX_c2.h5ad')
adat = sc.read_h5ad('../data/BMMC/ATAC_1.h5ad')
id_list = five_fold_split_dataset(rdat1)
train_id1, validation_id1, test_id1 = id_list[4]
train_r1, train_a= rdat1[train_id1+validation_id1,:], adat[train_id1+validation_id1,:]
id_list = five_fold_split_dataset(rdat2)
train_id2, validation_id2, test_id2 = id_list[4]
train_r2, train_t= rdat2[train_id2+validation_id2,:], tdat[train_id2+validation_id2,:]

In [3]:
from src.config import cfg
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Stage 1: Joint RNA–ATAC Training

Using paired scRNA-seq and scATAC-seq data, we jointly train RNA and ATAC conditional encoders. In the aligned space, contrastive learning together with a DEC-style clustering objective maximizes the mutual information between representations of the same cell across modalities, thereby establishing a core feature space.

In [4]:

step1 = ["atac","rna"]
cfg_rna_atac = {xtype: cfg[xtype] for xtype in step1}
scbridi_rna_atac = scBriDi(device=device, cfg=cfg_rna_atac, xtypes=step1, n_clusters=22,save_path='../param/brid').to(device)

scbridi_rna_atac.svd({"rna":train_r1,"atac":train_a})

Created features and saved for rna
Saved cell_type labels for atac modality
Created features and saved for atac


In [5]:

scbridi_rna_atac.lr =3e-5
scbridi_rna_atac.phases = {
                'pretrain': {'contrast': 0.0, "intra":0.0,"dec":0},
                'warmup': {'contrast': 0.05, "intra": 0.5, "dec": 0},
                'full': {'contrast': 0.3, "intra": 1,"dec": 0.4}}
scbridi_rna_atac.train_model(n_epoch=900)

100%|██████████| 10/10 [08:07<00:00, 48.71s/it, Phase: full | Loss: 2.2430 | contrastLoss: 2.2853 | intraLoss: 0.0202 | DECLoss: 0.0187 | atac: 0.0475 | rna: 0.1747 | contrastWeight: 0.283 | intraWeight: 0.967 | decWeight: 0.373]   


## Stage 2: RNA-Anchored ADT Alignment

We freeze the RNA conditional encoder trained in Stage 1 as a fixed semantic anchor. We then introduce a new modality — antibody-derived tags (ADT) — and train only the ADT conditional encoder with paired RNA–ADT data. Through contrastive learning against the frozen RNA encoder, ADT embeddings are aligned to the same RNA-centered semantic space.

In [6]:
step2 = ["rna","adt"]
cfg_rna_adt = {xtype: cfg[xtype] for xtype in step2}
scbridi_rna_adt = scBriDi(device=device, cfg=cfg_rna_adt, xtypes=step2,n_clusters=45,save_path='../param/brid').to(device)
scbridi_rna_adt.svd({"rna":train_r2,"adt":train_t})

Created features and saved for rna
Saved cell_type labels for adt modality
Created features and saved for adt


In [8]:
scbridi_rna_adt.load_model(param_dict={"rna":"../param/brid/model_rna.pth"},freeze=True)
scbridi_rna_adt.lr = 4e-5
scbridi_rna_adt.phases = {
                'pretrain': {'contrast': 0.0, "intra":0.0,"dec":0},
                'warmup': {'contrast': 0.1, "intra": 0.5, "dec": 0},
                'full': {'contrast': 0.3, "intra": 1,"dec": 0.3}}
scbridi_rna_adt.train_model(n_epoch=900)

100%|██████████| 10/10 [08:29<00:00, 50.95s/it, Phase: full | Loss: 5.0994 | contrastLoss: 4.8520 | intraLoss: 1.3984 | DECLoss: 1.1796 | rna: 0.1976 | adt: 0.3715 | contrastWeight: 0.287 | intraWeight: 0.967 | decWeight: 0.280]   


In [4]:
step3 = ["atac","adt"]

scbridi = scBriDi(device=device, cfg=cfg, xtypes=step3,save_path='../param/brid').to(device)
scbridi.load_model(param_dict={"adt":"../param/brid/model_adt.pth","atac":"../param/brid/model_atac.pth"})

## Cross-Modal Generation

With both ADT and ATAC encoders aligned to the same RNA-centered semantic space, we can perform cross-modal generation (e.g., ADT→ATAC and ATAC→ADT) without requiring direct ADT–ATAC paired data during training. This design reduces reliance on fully paired datasets and scales efficiently to future modalities.

In [7]:
p_atac = scbridi.inference(out_types=["atac"],condition_data={"adt":tdat[test_id2,:]})["atac"]



['adt'] pretrained


d:\Lab\code\scBriDi\tutorial\..\src\sampler.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  gt = torch.tensor(gt).to(device)
time: 0: 100%|██████████| 1000/1000 [13:40<00:00,  1.22it/s] 


In [8]:

sc.pp.pca(p_atac)
sc.pp.neighbors(p_atac)
sc.tl.tsne(p_atac)
sc.tl.leiden(p_atac)
result = cluster_metrics(p_atac)
print('ADT->ATAC generation:\nARI: %.3f, \tNMI: %.3f' % (result["ARI"], result["NMI"]))

d:\App\Anaconda\Lib\site-packages\scanpy\preprocessing\_pca\__init__.py:245: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  Version(ad.__version__) < Version("0.9")
d:\App\Anaconda\Lib\site-packages\scanpy\neighbors\__init__.py:427: FutureWarning: Use obsm (e.g. `k in adata.obsm` or `adata.obsm.keys() | {'u'}`) instead of AnnData.obsm_keys, AnnData.obsm_keys is deprecated and will be removed in the future.
  if "X_diffmap" in adata.obsm_keys():


adt2atac :
ARI: 0.006, 	NMI: 0.008


In [5]:
p_adt = scbridi.inference(out_types=["adt"],condition_data={"atac":adat[test_id1,:]})["adt"]

d:\Lab\code\scBriDi\tutorial\..\src\sampler.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  gt = torch.tensor(gt).to(device)
time: 0: 100%|██████████| 1000/1000 [10:38<00:00,  1.57it/s] 


In [6]:
sc.pp.pca(p_adt)
sc.pp.neighbors(p_adt)
sc.tl.tsne(p_adt)
sc.tl.leiden(p_adt)
result = cluster_metrics(p_adt)
print('ATAC->ADT generation:\nARI: %.3f, \tNMI: %.3f' % (result["ARI"], result["NMI"]))

d:\App\Anaconda\Lib\site-packages\scanpy\preprocessing\_pca\__init__.py:245: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  Version(ad.__version__) < Version("0.9")
d:\App\Anaconda\Lib\site-packages\scanpy\neighbors\__init__.py:427: FutureWarning: Use obsm (e.g. `k in adata.obsm` or `adata.obsm.keys() | {'u'}`) instead of AnnData.obsm_keys, AnnData.obsm_keys is deprecated and will be removed in the future.
  if "X_diffmap" in adata.obsm_keys():
C:\Users\Z\AppData\Local\Temp\ipykernel_20284\672612956.py:4: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(p_adt)


atac2adt :
ARI: 0.001, 	NMI: 0.005
